# 10 — Riesgo en los 6 distritos de Lima
Aplica los champions a los edificios de Lima:
- **Efecto del modelo de daño** (`08_ml_dano`) en cada distrito para los sismos registrados y para el sismo de Callao 2025 escalado a 0.25 g, 0.45 g (diseño) y 0.70 g. Se indica qué porcentaje de edificios queda fuera del rango con el que se entrenó.
- **Perfil de riesgo** (`09_ml_clustering`) de cada edificio.
- **Ranking** de distritos y hexágonos H3 por daño esperado con las curvas de fragilidad.

## 1. Setup — imports

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

## 2. Catálogo, esquema y parámetros

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

## 3. Funciones de escritura

In [ ]:
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


def subir_pandas(lotes, tabla, esquema=None, preparar=None, particion=None, mb_lote=64):
    if isinstance(lotes, pd.DataFrame):
        lotes = [lotes]
    modo, n_lotes = "overwrite", 0
    for pdf in lotes:
        if len(pdf) == 0:
            continue
        partes = max(1, int(np.ceil(pdf.memory_usage(deep=True).sum() / 1e6 / mb_lote)))
        for idx in np.array_split(np.arange(len(pdf)), partes):
            sdf = a_spark(pdf.iloc[idx], esquema)
            if preparar is not None:
                sdf = preparar(sdf)
            escribir(sdf, tabla, particion, modo, contar=False)
            modo, n_lotes = "append", n_lotes + 1
    n = spark.table(tabla).count()
    print(f"  {tabla}: {n:,} filas ({n_lotes} lotes)")
    return n


def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append({"tabla": t.tableName, "filas": spark.table(t.tableName).count(), "mb": round((d["sizeInBytes"] or 0) / 1e6, 1)})
    return pd.DataFrame(filas)

## 4. Champions de daño y de perfiles

In [ ]:
def champion(caso):
    fila = (spark.table("ml_registro_modelos").filter((F.col("caso") == caso) & F.col("promovido"))
            .orderBy(F.col("version").desc()).first())
    if fila is None:
        raise RuntimeError(f"No hay champion de {caso}: ejecute antes su notebook de ML")
    meta = {"version": int(fila["version"]), "algoritmo": fila["algoritmo"], "lote": fila["lote"],
            "cols_num": json.loads(fila["columnas_num"]), "metricas": json.loads(fila["metricas"]), "detalle": json.loads(fila["detalle"])}
    print(f"Champion de {caso}: v{meta['version']} ({meta['algoritmo']}, lote {meta['lote']})")
    return PipelineModel.load(fila["ruta"]), meta


modelo_dano, meta_dano = champion("dano")
modelo_clu, meta_clu = champion("clustering")
EVENTO_EFECTO = "2025-06-15_callao_M6.1"
COBERTURA_KM = 10.0

## 5. Efecto del modelo de daño en los 6 distritos
El modelo se entrenó en Tainan y Manta: en Lima no hay daño observado para validarlo. Por eso se reporta qué porcentaje de edificios queda fuera del rango de cada variable de entrenamiento.

In [ ]:
VARS_ONDA = ["PGA_g", "PGV_cms", "Sa03_g", "Sa06_g", "Sa10_g", "Sa_Tb"]
RANGO_DANO = meta_dano["detalle"]["rangos"]


def variables_lima(df):
    return (df.withColumn("PGV_cms", F.col("pgv_cms"))
            .withColumn("suelo_cod", F.when(F.col("vs30") < 180, 3.0).when(F.col("vs30") < 500, 2.0).when(F.col("vs30") < 1500, 1.0).otherwise(0.0)))


def fuera_de_rango(columna):
    a, b = RANGO_DANO[columna]
    return ~F.col(columna).between(a, b)


def efecto_por_distrito(df, claves):
    pred = modelo_dano.transform(variables_lima(df)).withColumn("indice_dano", vector_to_array("probability")[1])
    return (pred.groupBy(*claves).agg(
        F.count("*").alias("edificios"),
        F.expr("percentile_approx(PGA_g, 0.5)").alias("PGA_mediana_g"),
        F.expr("percentile_approx(Sa_Tb, 0.5)").alias("Sa_Tb_mediana_g"),
        F.avg("indice_dano").alias("indice_dano_medio"),
        (100 * F.avg((F.col("indice_dano") > 0.5).cast("double"))).alias("pct_indice_mayor_05"),
        *[(100 * F.avg(fuera_de_rango(c).cast("double"))).alias(f"pct_fuera_rango_{c}") for c in VARS_ONDA if c in RANGO_DANO]))

In [ ]:
escenarios_efecto = (spark.table("riesgo_escenarios")
                     .where((F.col("tipo") == "registrado") | (F.col("evento") == EVENTO_EFECTO))
                     .select("escenario", "evento", "tipo", "pga_roca_escenario_g").toPandas())
area_lima = spark.table("gold_edificios").select("id", "area_m2")
dem_efecto = (spark.table("riesgo_demanda").where(F.col("escenario").isin(escenarios_efecto.escenario.tolist()))
              .join(area_lima, "id"))
escribir(efecto_por_distrito(dem_efecto, ["escenario", "evento", "tipo", "ubigeo", "distrito"]), "riesgo_efecto_lima")
efecto = spark.table("riesgo_efecto_lima").toPandas()
display(efecto.pivot_table(index="distrito", columns="escenario", values="indice_dano_medio").round(3))
display(efecto.groupby("escenario")[["PGA_mediana_g"] + [c for c in efecto.columns if c.startswith("pct_fuera_rango_")]].mean().round(3))

## 6. Perfil de riesgo de cada edificio

In [ ]:
lote_clustering = json.loads(spark.table("riesgo_parametros").where("clave = 'lote_clustering'").first()["valor"])
df_km = spark.table("fe_clustering").where(F.col("lote") == lote_clustering)
orden = {int(k): int(v) for k, v in meta_clu["detalle"]["orden"].items()}
mapa = F.create_map(*[F.lit(x) for par in orden.items() for x in par])
cluster_edificio = (modelo_clu.transform(df_km).withColumn("cluster", mapa[F.col("cluster")])
                    .select("id", "cluster", F.lit(meta_clu["version"]).alias("version_modelo")))
escribir(cluster_edificio, "riesgo_cluster_edificio")

KM_COLS = ["ln_Sa_Tb", "ln_IM_theta", "Tb_TP", "vs30", "ln_pisos"]
perfiles = (df_km.join(spark.table("riesgo_cluster_edificio"), "id").groupBy("cluster")
            .agg(*[F.avg(c).alias(c) for c in KM_COLS], F.count("*").alias("edificios"),
                 F.avg("P_colapso").alias("P_colapso_media"), F.avg("dano_esperado").alias("dano_esperado_medio"),
                 *[(100 * F.avg((F.col("tipologia") == t).cast("double"))).alias(f"pct_{t.split('_')[0]}")
                   for t in ["adobe", "quincha", "madera", "concreto_armado"]])
            .orderBy("cluster").toPandas())
perfiles["Sa_Tb_g"] = np.exp(perfiles.ln_Sa_Tb)
perfiles["IM_sobre_theta_colapso"] = np.exp(perfiles.ln_IM_theta)
perfiles["pisos"] = np.exp(perfiles.ln_pisos)
perfiles["silueta"] = meta_clu["metricas"]["Silhouette"]
subir_pandas(perfiles, "riesgo_perfiles")
display(perfiles.drop(columns=["ln_Sa_Tb", "ln_IM_theta", "ln_pisos"]).round(3))

## 7. Ranking de distritos y hexágonos H3

In [ ]:
de = spark.table("riesgo_dano_edificios")
UMBRAL_COLAPSO = 0.10
MIN_EDIFICIOS_RANKING = 1000
MAX_PCT_FUERA_COBERTURA = 50.0


def metricas_riesgo(df, claves):
    return df.groupBy(*claves).agg(
        F.count("*").alias("edificios"),
        F.expr("percentile_approx(Sa_Tb, 0.5)").alias("Sa_Tb_mediana_g"),
        F.avg("P_colapso").alias("P_colapso_media"),
        F.avg("P_colapso_bajo").alias("P_colapso_media_bajo"), F.avg("P_colapso_alto").alias("P_colapso_media_alto"),
        (100 * F.avg(F.col("fuera_cobertura").cast("double"))).alias("pct_fuera_cobertura"),
        (100 * F.avg((F.col("P_colapso") > UMBRAL_COLAPSO).cast("double"))).alias("pct_P_colapso_mayor_umbral"),
        F.avg("dano_esperado").alias("dano_esperado_medio"),
        (100 * F.avg((F.abs(F.col("Tb_TP") - 1) < 0.2).cast("double"))).alias("pct_resonancia"),
        F.avg("vs30").alias("vs30_medio"), F.avg("pisos").alias("pisos_medio"))


ranking = (metricas_riesgo(de, ["escenario", "evento", "tipo", "ubigeo", "distrito", "provincia"])
           .withColumn("muestra_suficiente", F.col("edificios") >= MIN_EDIFICIOS_RANKING)
           .withColumn("cobertura_suficiente", F.col("pct_fuera_cobertura") < MAX_PCT_FUERA_COBERTURA)
           .withColumn("rankeable", F.col("muestra_suficiente") & F.col("cobertura_suficiente")))
orden = Window.partitionBy("escenario", "rankeable").orderBy(F.desc("dano_esperado_medio"))
ranking = ranking.withColumn("ranking", F.when(F.col("rankeable"), F.rank().over(orden)))
escribir(ranking, "riesgo_ranking_distritos")

centros_h3 = spark.table("gold_edificios").groupBy("h3_r8").agg(F.avg("lon").alias("lon"), F.avg("lat").alias("lat"))
escribir(metricas_riesgo(de, ["escenario", "evento", "tipo", "h3_r8"]).join(centros_h3, "h3_r8", "left"), "riesgo_h3", particion="escenario")

ESCENARIO_VISTA = "e030_sitio"
rk = (spark.table("riesgo_ranking_distritos").where((F.col("escenario") == ESCENARIO_VISTA) & F.col("ranking").isNotNull())
      .orderBy("ranking").toPandas())
display(rk.head(10)[["ranking", "distrito", "provincia", "edificios", "Sa_Tb_mediana_g", "P_colapso_media", "P_colapso_media_bajo",
                     "P_colapso_media_alto", "dano_esperado_medio", "pct_resonancia", "vs30_medio", "pct_fuera_cobertura"]])
fuera = spark.table("riesgo_ranking_distritos").where((F.col("escenario") == ESCENARIO_VISTA) & ~F.col("rankeable")).toPandas()
print(f"Sin ranking en {ESCENARIO_VISTA}: menos de {MIN_EDIFICIOS_RANKING} edificios ->",
      fuera.loc[~fuera.muestra_suficiente, "distrito"].tolist(),
      f"| más de {MAX_PCT_FUERA_COBERTURA:.0f} % de edificios a más de {COBERTURA_KM:.0f} km de una estación ->",
      fuera.loc[fuera.muestra_suficiente & ~fuera.cobertura_suficiente, "distrito"].tolist())

h3v = spark.table("riesgo_h3").where(F.col("escenario") == ESCENARIO_VISTA).toPandas()
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
top = rk.head(15).iloc[::-1]
ax[0].barh(top.distrito, top.dano_esperado_medio)
ax[0].set(xlabel="Daño esperado medio (0-4)", title=f"Distritos más afectados — {ESCENARIO_VISTA}")
sc = ax[1].scatter(h3v.lon, h3v.lat, c=h3v.P_colapso_media, s=6, cmap="inferno_r")
plt.colorbar(sc, ax=ax[1], shrink=.7, label="P(colapso) media")
ax[1].set_title("P(colapso) media por hexágono H3 r8"); ax[1].set_aspect("equal")
plt.tight_layout(); plt.show()

## 8. Optimización y verificación

In [ ]:
for tabla in ["riesgo_demanda", "riesgo_dano_edificios"]:
    spark.sql(f"OPTIMIZE {tabla} ZORDER BY (h3_r8, distrito)")
display(inventario("riesgo_"))